# Target preset lookup example

Look up a QUEL product preset from a catalog SKU, unit serial, or fragment.
This is the notebook version of
`qal/examples/general/target_preset_lookup_example.py`.

**How to run**

1. Run the cells from top to bottom.
2. In *Look up a unique preset*, edit `TARGET_QUERY` (or keep the default).
3. Run *Find all matches* when the query is ambiguous.

Examples of valid queries: `RCS-ICG-ST01-QUEL03`, `RCS-S-IC1-S1-A-113`,
`RDS-80Q-LU05`, `RCS-ICG`.


## 1. Setup

This cell loads the lookup helpers.

`SAVE_RESULTS` writes the query, partial-match diagnostic (if any), and
returned presets to `target_lookup.yaml`. Set it to `False` to disable saving.
Malformed serials report recognized fields and unknown tokens; candidate
presets are not safe for analysis until the query is corrected.


In [ ]:
from qal.util import describe_target_query, find_target_presets, lookup_target_preset, print_preset
from pathlib import Path
import yaml

SAVE_RESULTS = True
OUTPUT_DIR = Path("target_preset_lookup_results")

def save_lookup(query, matches):
    if SAVE_RESULTS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        (OUTPUT_DIR / "target_lookup.yaml").write_text(yaml.safe_dump({
            "query": query, "diagnostic": describe_target_query(query),
            "matches": matches,
        }, sort_keys=False), encoding="utf-8")


## 2. Look up a unique preset

Set `TARGET_QUERY` to a catalog SKU, unit serial, unique fragment, or
family-scoped tag. `lookup_target_preset` requires the query to resolve to
exactly one product. Missing or ambiguous matches print a message without a
traceback. For multiple matches, try a more specific query or run the next
section instead.


In [ ]:
# Catalog SKU, unit serial, unique fragment, or family-scoped tag.
# Examples: "RCS-ICG-ST01-QUEL03", "RCS-S-IC1-S1-A-000", "RDS-80Q-LU05".
TARGET_QUERY = "V3-ICG-ST03-RDS000"

preset = None
try:
    preset = lookup_target_preset(TARGET_QUERY)
except ValueError as exc:
    print(exc)
    save_lookup(TARGET_QUERY, [])
else:
    print_preset(preset)
    save_lookup(TARGET_QUERY, [preset])


## 3. Find all matches

Use this when the query is a fragment such as `ICG` or `RCS-ICG` and you want
every matching preset printed. If no presets match, the cell prints a
message and finishes normally.


In [ ]:
TARGET_QUERY = "ICG"

matches = []
try:
    matches = find_target_presets(TARGET_QUERY)
except ValueError as exc:
    print(exc)
else:
    diagnostic = describe_target_query(TARGET_QUERY)
    if diagnostic:
        print(diagnostic)
    save_lookup(TARGET_QUERY, matches)
    if not matches:
        print(f"No product preset matches {TARGET_QUERY!r}")
    for match in matches:
        print_preset(match)
        print()
